## Exploratory Data Analysis on the raw data

In [1]:
# Import all the desired packages and libraries
import os

import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


- Load raw data from local directory

In [2]:
DATA_DIR = 'EPL_2025'

def load_match_data(data_path: str = DATA_DIR):
    unique_teams = set()
    match_data = []

    for dirpath, dirnames, filenames in os.walk(data_path):
        if "context.json" in filenames and "shots.json" in filenames:
            try:
                with open(os.path.join(dirpath, "context.json"), "r") as f:
                    context = json.load(f)
                with open(os.path.join(dirpath, "shots.json"), "r") as f:
                    shots = json.load(f)
                
                # Extract relevant data from context and shots files
                home_team = context.get("team_h")
                away_team = context.get("team_a")
                match_data.append({
                    "home_team": home_team,
                    "away_team": away_team,
                    "h_goals": int(context.get("h_goals")),
                    "a_goals": int(context.get("a_goals")),
                    "h_shot": int(context.get("h_shot")),
                    "a_shot": int(context.get("a_shot")),
                    "h_xg": float(context.get("h_xg")),
                    "a_xg": float(context.get("a_xg")),
                    "h_shotOnTarget": int(context.get("h_shotOnTarget")),
                    "a_shotOnTarget": int(context.get("a_shotOnTarget")),
                    "h_ppda": float(context.get("h_ppda")),
                    "a_ppda": float(context.get("a_ppda"))
                })
                unique_teams.add(home_team)
                unique_teams.add(away_team)
            except Exception as e:
                print(f"Error loading data from {dirpath}: {e}")
    sorted_teams = sorted(list(unique_teams))
    match_data = pd.DataFrame(match_data)

    return sorted_teams, match_data

teams, raw_data = load_match_data(DATA_DIR)
# if teams:
#     print(f"Loaded {len(teams)} unique teams: {teams}")
# if raw_data:
#     print(f"Loaded {len(raw_data)} matches.")
#     print(f"Sample data: {raw_data[0] if raw_data else 'No data available.'}")


- Select a team for Exploratory Data Analysis (EDA)

In [ ]:
team_selector = widgets.Dropdown(
    options=teams,
    value=None if teams else None,
    description='Select a team for EDA:',
    disabled=not bool(teams),
    style={'description_width': 'initial'}
)

canvas_output = widgets.Output()


def plot_team_xg_distribution(team, df):
    # Helper function to plot the xG distribution for a team.
    # This function will be evolved to include a complete EDA analysis.
    team_df = df[(df['home_team'] == team) | (df['away_team'] == team)]
    xgs = (list(team_df.loc[team_df['home_team'] == team, 'h_xg'].astype(float)) + list(team_df.loc[team_df['away_team'] == team, 'a_xg'].astype(float)))
    fig, ax = plt.subplots(figsize=(7,3))
    sns.histplot(xgs, bins=10, kde=False)
    plt.title(f'{team} xG distribution')
    display(fig)
    plt.close(fig)
    


def dropdown_handler(change):
    new_team = change['new']
    with canvas_output:
        if new_team not in teams:
            display(f"Invalid team selected: {new_team}")
            return
        
        canvas_output.clear_output()
        display(f"Performing EDA for {new_team}")
        plot_team_xg_distribution(new_team, raw_data)

try:
    team_selector.unobserve(dropdown_handler, names="value")
except ValueError:
    pass

team_selector.observe(dropdown_handler, names="value")

display(widgets.VBox([team_selector, canvas_output]))